# 3D Object Reconstruction with SAM-3D-Objects and OpenVINO

[SAM-3D-Objects](https://github.com/facebookresearch/sam-3d-objects) is Meta's state-of-the-art pipeline for lifting 2D images into 3D Gaussian Splats. Given a single image and an object mask, the pipeline reconstructs a full 3D representation of the object through a two-stage process:

1. **Stage 1 — Sparse Structure Generation**: DINOv2 condition embeddings drive a flow-matching transformer (SS Generator) to predict a sparse 3D occupancy grid, decoded by the SS Decoder into voxel coordinates.
2. **Stage 2 — Structured Latent Generation**: A second flow-matching transformer (SLat Generator) produces per-voxel latent features, which SLat Decoders convert into 3D Gaussian Splat parameters (position, opacity, scale, rotation, color).

The pipeline also uses **MoGe** (Monocular Geometry Estimation) for depth/point-map prediction and **PointPatchEmbed** for fusing geometric information with visual features.

In this notebook, we demonstrate how to:
- Download the SAM-3D-Objects model from ModelScope
- Convert all weighted sub-models to OpenVINO IR format
- Run single-object and multi-object 3D reconstruction with OpenVINO acceleration
- Optimize models using NNCF post-training quantization
- Build an interactive Gradio demo

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Download Model and Configure Pipeline](#Download-Model-and-Configure-Pipeline)
- [Convert SAM-3D Models to OpenVINO IR Format](#Convert-SAM-3D-Models-to-OpenVINO-IR-Format)
- [Select Inference Device](#Select-Inference-Device)
- [Verify Conversion Accuracy](#Verify-Conversion-Accuracy)
- [Create OpenVINO Pipeline](#Create-OpenVINO-Pipeline)
- [Single Object 3D Reconstruction](#Single-Object-3D-Reconstruction)
- [Multi Object 3D Reconstruction](#Multi-Object-3D-Reconstruction)
- [Optimize with NNCF Quantization](#Optimize-with-NNCF-Quantization)
- [Interactive Gradio Demo](#Interactive-Gradio-Demo)

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [1]:
# Fetch `notebook_utils` module
import requests
from pathlib import Path

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )
    open("notebook_utils.py", "w").write(r.text)

if not Path("cmd_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py",
    )
    open("cmd_helper.py", "w").write(r.text)

if not Path("pip_helper.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/pip_helper.py",
    )
    open("pip_helper.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("sam-3d-objects-reconstruction.ipynb")

In [3]:
from cmd_helper import clone_repo
from pip_helper import pip_install

# Install required dependencies for SAM-3D-Objects and OpenVINO
pip_install(
    "-q",
    "openvino>=2026.0.0",
    "nncf>=2.13",
    "gradio>=4.13",
    "torch",
    "torchvision",
    "numpy",
    "Pillow",
    "matplotlib",
    "omegaconf",
    "hydra-core",
    "trimesh",
    "imageio",
    "scipy",
    "einops",
    "roma",
    "rootutils",
    "astor",
    "easydict",
    "lightning",
    "plyfile",
    "pyvista",
    "scikit-image",
    "opencv-python",
    "igraph",
    "modelscope",
)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
openvino-tokenizers 2025.4.0.0 requires openvino~=2025.4.0.dev, but you have openvino 2026.0.0 which is incompatible.


In [4]:
import os
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import torch

NOTEBOOK_DIR = Path.cwd()

# ---------------------------------------------------------------------------
# Download SAM-3D-Objects model weights from ModelScope
# ---------------------------------------------------------------------------
from modelscope import snapshot_download

SAM3D_MODEL_ROOT = Path(
    snapshot_download(
        "facebook/sam-3d-objects",
        cache_dir=str(NOTEBOOK_DIR / "models"),
    )
)
print(f"Model downloaded to: {SAM3D_MODEL_ROOT}")

# ---------------------------------------------------------------------------
# Clone SAM-3D-Objects source code (inference pipeline)
# ---------------------------------------------------------------------------
SAM3D_ROOT = NOTEBOOK_DIR / "sam-3d-objects"
if not (SAM3D_ROOT / "sam3d_objects").exists():
    import subprocess
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/facebookresearch/sam-3d-objects.git", str(SAM3D_ROOT)],
        check=True,
    )
    print(f"Cloned sam-3d-objects repo to: {SAM3D_ROOT}")
else:
    print(f"sam-3d-objects repo already exists at: {SAM3D_ROOT}")

SAM3D_NOTEBOOK = SAM3D_ROOT / "notebook"

for p in [str(SAM3D_ROOT), str(SAM3D_NOTEBOOK), str(NOTEBOOK_DIR)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Set environment variables BEFORE any sam3d_objects imports
os.environ.setdefault("CONDA_PREFIX", "/usr")
os.environ.setdefault("CUDA_HOME", "/usr")

# Change working directory so Hydra relative paths work
os.chdir(str(SAM3D_ROOT))

print(f"SAM-3D-Objects root: {SAM3D_ROOT}")
print(f"Model checkpoints:   {SAM3D_MODEL_ROOT}")
print(f"Notebook directory:  {NOTEBOOK_DIR}")
print(f"PyTorch version:     {torch.__version__}")
print(f"CUDA available:      {torch.cuda.is_available()}")

2026-03-11 22:44:01,513 - modelscope - INFO - Not logged-in, you can login for uploadingor accessing controlled entities.


2026-03-11 22:44:05,520 - modelscope - INFO - Got 28 files, start to download ...


Processing 28 items:   0%|          | 0.00/28.0 [00:00<?, ?it/s]

2026-03-11 22:54:21,155 - modelscope - INFO - Download model 'facebook/sam-3d-objects' successfully.


Model downloaded to: /home/ethan/intel/openvino_notebooks/notebooks/sam-3d-objects-reconstruction/models/facebook/sam-3d-objects


Cloning into '/home/ethan/intel/openvino_notebooks/notebooks/sam-3d-objects-reconstruction/sam-3d-objects'...


Cloned sam-3d-objects repo to: /home/ethan/intel/openvino_notebooks/notebooks/sam-3d-objects-reconstruction/sam-3d-objects
SAM-3D-Objects root: /home/ethan/intel/openvino_notebooks/notebooks/sam-3d-objects-reconstruction/sam-3d-objects
Model checkpoints:   /home/ethan/intel/openvino_notebooks/notebooks/sam-3d-objects-reconstruction/models/facebook/sam-3d-objects
Notebook directory:  /home/ethan/intel/openvino_notebooks/notebooks/sam-3d-objects-reconstruction
PyTorch version:     2.10.0+cpu
CUDA available:      False


In [5]:
# Apply CUDA patches — this MUST happen before any sam3d_objects import.
# The helper mocks CUDA-only packages (pytorch3d, spconv, kaolin) and patches
# the pipeline to run on CPU.
import sam_3d_objects_helper as helper

helper.patch_cuda_for_cpu()

2026-03-11 22:54:25.854 | INFO     | sam3d_objects.pipeline.inference_pipeline:set_attention_backend:17 - GPU name is CPU


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


2026-03-11 22:54:28.701 | INFO     | sam3d_objects.model.backbone.tdfy_dit.modules.sparse:__from_env:39 - [SPARSE] Backend: spconv, Attention: sdpa
2026-03-11 22:54:28.706 | INFO     | sam3d_objects.model.backbone.tdfy_dit.modules.attention:__from_env:30 - [ATTENTION] Using backend: sdpa


[SPARSE][CONV] spconv algo: auto


2026-03-11 22:54:30.091 | WARNING  | sam3d_objects.data.dataset.tdfy.preprocessor:__post_init__:51 - No rgb pointmap normalizer provided, using scale + shift 
2026-03-11 22:54:30.092 | WARNING  | sam3d_objects.data.dataset.tdfy.preprocessor:__post_init__:51 - No rgb pointmap normalizer provided, using scale + shift 


[OV-SAM3D] CUDA patches applied — running on CPU / OpenVINO


## Download Model and Configure Pipeline
[back to top ⬆️](#Table-of-contents:)

The SAM-3D-Objects pipeline is configured via a Hydra YAML config (`pipeline.yaml`) that specifies:
- **Condition embedders**: DINOv2 ViT-L/14 backbones for image and mask encoding
- **SS Generator**: 24-block MOT (Multi-Object Transformer) for sparse structure flow matching
- **SS Decoder**: 3D convolutional VAE decoder (latent → occupancy grid)
- **SLat Generator**: 24-block sparse transformer for structured latent flow matching
- **SLat Decoders**: Transformer-based decoders for Gaussian splat parameters
- **MoGe**: ViT-based monocular geometry estimator for point-map prediction

We load the config, patch it for CPU inference, and instantiate the full pipeline.

In [6]:
from omegaconf import OmegaConf
from hydra.utils import instantiate

CONFIG_PATH = SAM3D_MODEL_ROOT / "checkpoints" / "pipeline.yaml"
assert CONFIG_PATH.exists(), f"pipeline.yaml not found at {CONFIG_PATH}"

# Load and patch the pipeline config for CPU inference
config = OmegaConf.load(str(CONFIG_PATH))
config = helper.patch_pipeline_config(config)
config.workspace_dir = str(CONFIG_PATH.parent)

print("Pipeline configuration:")
print(f"  Device:           {config.device}")
print(f"  Dtype:            {config.dtype}")
print(f"  Decode formats:   {config.decode_formats}")
print(f"  Rendering engine: {config.rendering_engine}")

Pipeline configuration:
  Device:           cpu
  Dtype:            float32
  Decode formats:   ['gaussian']
  Rendering engine: pytorch3d


In [10]:
# Instantiate the full pipeline on CPU
# This loads all model weights (DINOv2, SS Generator, SLat Generator, decoders, MoGe)
print("Instantiating SAM-3D-Objects pipeline on CPU …")
t0 = time.time()

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    pipeline = instantiate(config)

t1 = time.time()
print(f"Pipeline loaded in {t1 - t0:.1f}s on device: {pipeline.device}")

# Print model summary
print(f"\nLoaded models:")
for name in pipeline.models:
    model = pipeline.models[name]
    n_params = sum(p.numel() for p in model.parameters()) if hasattr(model, "parameters") else 0
    print(f"  {name:30s}  {n_params / 1e6:>8.2f}M params")

print(f"\nCondition embedders:")
for name in pipeline.condition_embedders:
    print(f"  {name}")

2026-03-11 22:55:36.895 | WARNING  | sam3d_objects.data.dataset.tdfy.preprocessor:__post_init__:51 - No rgb pointmap normalizer provided, using scale + shift 
2026-03-11 22:55:36.900 | WARNING  | sam3d_objects.data.dataset.tdfy.preprocessor:__post_init__:51 - No rgb pointmap normalizer provided, using scale + shift 
2026-03-11 22:55:36.902 | INFO     | sam3d_objects.pipeline.inference_pipeline:__init__:100 - self.device: cpu
2026-03-11 22:55:36.903 | INFO     | sam3d_objects.pipeline.inference_pipeline:__init__:101 - CUDA_VISIBLE_DEVICES: None
2026-03-11 22:55:36.904 | INFO     | sam3d_objects.pipeline.inference_pipeline:__init__:102 - Actually using GPU: cpu(mock)
2026-03-11 22:55:36.904 | INFO     | sam3d_objects.pipeline.inference_pipeline:init_pose_decoder:297 - Using pose decoder: ScaleShiftInvariant
2026-03-11 22:55:36.905 | INFO     | sam3d_objects.pipeline.inference_pipeline:__init__:133 - Loading model weights...


Instantiating SAM-3D-Objects pipeline on CPU …


2026-03-11 22:55:45.344 | INFO     | sam3d_objects.model.io:load_model_from_checkpoint:158 - Loading checkpoint from /home/ethan/intel/openvino_notebooks/notebooks/sam-3d-objects-reconstruction/models/facebook/sam-3d-objects/checkpoints/ss_generator.ckpt
2026-03-11 22:55:53.464 | INFO     | sam3d_objects.model.io:load_model_from_checkpoint:158 - Loading checkpoint from /home/ethan/intel/openvino_notebooks/notebooks/sam-3d-objects-reconstruction/models/facebook/sam-3d-objects/checkpoints/slat_generator.ckpt
2026-03-11 22:55:56.100 | INFO     | sam3d_objects.model.io:load_model_from_checkpoint:158 - Loading checkpoint from /home/ethan/intel/openvino_notebooks/notebooks/sam-3d-objects-reconstruction/models/facebook/sam-3d-objects/checkpoints/ss_decoder.ckpt
2026-03-11 22:55:56.942 | INFO     | sam3d_objects.model.io:load_model_from_checkpoint:158 - Loading checkpoint from /home/ethan/intel/openvino_notebooks/notebooks/sam-3d-objects-reconstruction/models/facebook/sam-3d-objects/checkpoint

Pipeline loaded in 42.1s on device: cpu

Loaded models:
  ss_generator                      959.96M params
  slat_generator                    600.43M params
  ss_encoder                          0.00M params
  ss_decoder                         73.67M params
  slat_decoder_gs                    85.37M params
  slat_decoder_gs_4                  85.07M params
  slat_decoder_mesh                  90.93M params

Condition embedders:
  ss_condition_embedder
  slat_condition_embedder


## Select Inference Device
[back to top ⬆️](#Table-of-contents:)

Select the device to use for OpenVINO inference. The device will be used for both model compilation and inference. Available devices depend on your system configuration.

In [11]:
from notebook_utils import device_widget

device = device_widget()
device

Dropdown(description='Device:', index=1, options=('CPU', 'AUTO'), value='AUTO')

## Convert SAM-3D Models to OpenVINO IR Format
[back to top ⬆️](#Table-of-contents:)

Now we convert all weighted sub-models in the pipeline to OpenVINO Intermediate Representation (IR) format. The conversion process:

1. **Wraps** each PyTorch module in a clean `nn.Module` designed for static-graph export (no dynamic control flow, no `SparseTensor`)
2. **Converts** using `ov.convert_model()` with example inputs
3. **Saves** the IR files (`.xml` + `.bin`) to disk

> **DINOv2 Backbone Merging**: All four DINOv2 embedders (SS-image, SS-mask, SLat-image, SLat-mask) share **identical** ViT-L/14 backbone weights (~1.2 GB each). We merge them into a **single** OV model with two outputs (postnorm for SS, prenorm for SLat), saving ~3.6 GB of disk space.

The following models are converted:

| Model | Architecture | Purpose |
|-------|-------------|---------|
| DINOv2 Backbone (merged) | ViT-L/14 (2 outputs) | Shared image/mask condition encoding for SS & SLat stages |
| SS Decoder | 3D ConvNet | Latent volume → occupancy grid |
| SS Generator | 24-block MOT Transformer | Sparse structure flow matching backbone |
| SLat Generator Core | 24-block Sparse Transformer | Structured latent flow matching backbone |
| SLat Decoders (GS/GS-4) | 12-block Sparse Transformer | Per-voxel Gaussian splat parameter prediction |
| SLat Mesh Decoder (base) | 12-block Sparse Transformer | Per-voxel mesh features (upsample + FlexiCubes stays in PyTorch) |
| MoGe | ViT | Monocular geometry estimation |
| Embedder Projections | LayerNorm + FFN | Per-embedder feature projection |

In [ ]:
OV_MODEL_DIR = NOTEBOOK_DIR / "ov_models"
OV_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Convert all sub-models to OpenVINO IR
print(f"Converting all models to OpenVINO IR (device={device.value}) …\n")
t0 = time.time()

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    compiled_models = helper.convert_all_models(pipeline, OV_MODEL_DIR, device=device.value)

t1 = time.time()
print(f"\nAll conversions completed in {t1 - t0:.1f}s")
print(f"Converted {len(compiled_models)} models")

# List saved IR files
print(f"\nSaved IR files in {OV_MODEL_DIR}:")
for f in sorted(OV_MODEL_DIR.glob("*.xml")):
    bin_file = f.with_suffix(".bin")
    bin_size = bin_file.stat().st_size / (1024 * 1024) if bin_file.exists() else 0
    print(f"  {f.name:40s}  {bin_size:>8.1f} MB")

Converting all models to OpenVINO IR (device=AUTO) …

[OV-SAM3D] Converting merged DINOv2 backbone (1 model for all 4 embedders) …
[OV-SAM3D] Saved merged DINOv2 backbone → /home/ethan/intel/openvino_notebooks/notebooks/sam-3d-objects-reconstruction/ov_models/dino_backbone.xml
[OV-SAM3D] Converting SS decoder …
[OV-SAM3D] Saved SS decoder → /home/ethan/intel/openvino_notebooks/notebooks/sam-3d-objects-reconstruction/ov_models/ss_decoder.xml
[OV-SAM3D] Converting SS generator backbone …


## Verify Conversion Accuracy
[back to top ⬆️](#Table-of-contents:)

To ensure the OpenVINO conversion is numerically faithful, we compare the outputs of the original PyTorch models with their OpenVINO counterparts using synthetic inputs. We measure:
- **Max absolute difference** — worst-case numerical error
- **Mean absolute difference** — average numerical error
- **Cosine similarity** — directional alignment of output vectors (1.0 = perfect)

In [ ]:
core = ov.Core()
accuracy_results = []

# --- Merged DINOv2 Backbone (both outputs) ---
print("=" * 60)
print("Merged DINOv2 Backbone (postnorm & prenorm outputs)")
print("=" * 60)
ss_emb = pipeline.condition_embedders["ss_condition_embedder"]
ss_dino_image = ss_emb.embedder_list[0][0]
ss_dino_mask = ss_emb.embedder_list[1][0]

# Load merged backbone
ov_model = core.read_model(str(OV_MODEL_DIR / "dino_backbone.xml"))
ov_compiled = core.compile_model(ov_model, device.value)

# Test 1: 3-channel image input → postnorm output (output 0, for SS stage)
test_input = torch.randn(1, 3, 518, 518, dtype=torch.float32)
with torch.no_grad():
    pt_out = ss_dino_image(test_input)  # SS uses prenorm=False → postnorm

ov_result = ov_compiled(test_input.numpy())
ov_out = torch.from_numpy(ov_result[0].copy())  # output 0 = postnorm
ok1 = helper.compare_outputs(pt_out, ov_out, "DINOv2 Backbone (postnorm / SS Image)", atol=0.02)
accuracy_results.append(("DINOv2 Backbone postnorm", ok1, pt_out.shape))

# Test 2: 1-channel mask input → postnorm output (output 0)
test_mask_input = torch.randn(1, 1, 518, 518, dtype=torch.float32)
with torch.no_grad():
    pt_out_mask = ss_dino_mask(test_mask_input)  # expects 1ch, repeats internally

# Merged backbone expects 3ch — repeat here like OVDinoEmbedder does
mask_3ch = test_mask_input.repeat(1, 3, 1, 1)
ov_result_mask = ov_compiled(mask_3ch.numpy())
ov_out_mask = torch.from_numpy(ov_result_mask[0].copy())  # postnorm
ok2 = helper.compare_outputs(pt_out_mask, ov_out_mask, "DINOv2 Backbone (postnorm / SS Mask)", atol=0.02)
accuracy_results.append(("DINOv2 Backbone postnorm (mask)", ok2, pt_out_mask.shape))

# Test 3: prenorm output (output 1, for SLat stage)
slat_emb = pipeline.condition_embedders["slat_condition_embedder"]
slat_dino_image = slat_emb.embedder_list[0][0]
with torch.no_grad():
    pt_out_slat = slat_dino_image(test_input)  # SLat uses prenorm=True

ov_out_prenorm = torch.from_numpy(ov_result[1].copy())  # output 1 = prenorm
ok3 = helper.compare_outputs(pt_out_slat, ov_out_prenorm, "DINOv2 Backbone (prenorm / SLat Image)", atol=0.02)
accuracy_results.append(("DINOv2 Backbone prenorm", ok3, pt_out_slat.shape))

# --- SS Decoder ---
print("\n" + "=" * 60)
print("SS Decoder (3D ConvNet)")
print("=" * 60)
ss_decoder = pipeline.models["ss_decoder"]

test_latent = torch.randn(1, 8, 16, 16, 16, dtype=torch.float32)
with torch.no_grad():
    pt_dec = ss_decoder(test_latent)

ov_model = core.read_model(str(OV_MODEL_DIR / "ss_decoder.xml"))
ov_compiled_dec = core.compile_model(ov_model, device.value)
ov_dec = torch.from_numpy(ov_compiled_dec(test_latent.numpy())[0].copy())

ok4 = helper.compare_outputs(pt_dec, ov_dec, "SS Decoder", atol=5e-3)
accuracy_results.append(("SS Decoder", ok4, pt_dec.shape))

# --- Summary ---
print("\n" + "=" * 60)
print("Accuracy Verification Summary")
print("=" * 60)
for name, passed, shape in accuracy_results:
    status = "PASS ✓" if passed else "FAIL ✗"
    print(f"  {status}  {name:35s}  output shape: {shape}")

## Create OpenVINO Pipeline
[back to top ⬆️](#Table-of-contents:)

Create the OpenVINO-accelerated pipeline by replacing **all** weighted sub-models with their OV compiled counterparts (compiled for the selected device):

- 4× DINOv2 embedders (SS image/mask, SLat image/mask) → `OVDinoEmbedder`
- SS Decoder → `OVSSDecoder`
- SS Generator backbone → monkey-patched to call OV model
- SLat Generator backbone → monkey-patched with U-Net bridge (input/output blocks in PyTorch, 24 core transformer blocks in OV)
- SLat Decoders (GS, GS-4) → monkey-patched to call OV models
- SLat Mesh Decoder → hybrid: transformer base in OV, upsample (SparseSubdivide) + FlexiCubes mesh extraction in PyTorch
- MoGe → `OVMoGe`

After this step, every forward pass through a weighted model will use OpenVINO inference. The pipeline produces both **Gaussian Splat** and **mesh** outputs simultaneously.

In [ ]:
# Create the OV-accelerated pipeline
print("Creating OpenVINO-accelerated pipeline …\n")

ov_pipeline = helper.OVInferencePipelinePointMap(pipeline, compiled_models)

# Verify that models were replaced
ss_emb = pipeline.condition_embedders["ss_condition_embedder"]
slat_emb = pipeline.condition_embedders["slat_condition_embedder"]

print("\nModel replacement status:")
print(f"  SS image embedder:   {'OV ✓' if isinstance(ss_emb.embedder_list[0][0], helper.OVDinoEmbedder) else 'PyTorch'}")
print(f"  SS mask embedder:    {'OV ✓' if isinstance(ss_emb.embedder_list[1][0], helper.OVDinoEmbedder) else 'PyTorch'}")
print(f"  SLat image embedder: {'OV ✓' if isinstance(slat_emb.embedder_list[0][0], helper.OVDinoEmbedder) else 'PyTorch'}")
print(f"  SLat mask embedder:  {'OV ✓' if isinstance(slat_emb.embedder_list[1][0], helper.OVDinoEmbedder) else 'PyTorch'}")
print(f"  SS decoder:          {'OV ✓' if isinstance(pipeline.models['ss_decoder'], helper.OVSSDecoder) else 'PyTorch'}")

## Single Object 3D Reconstruction
[back to top ⬆️](#Table-of-contents:)

Following the original [`demo_single_object.ipynb`](https://github.com/facebookresearch/sam-3d-objects/blob/main/notebook/demo_single_object.ipynb), we load a single image and one object mask, then run the **full** two-stage OpenVINO pipeline to produce both a **3D Gaussian Splat** and a **vertex-colored mesh (GLB)**.

1. Load image + mask (index 14 — a piece of furniture from a kids' room scene)
2. Run Stage 1 (Sparse Structure Generation) → voxel coordinates
3. Run Stage 2 (Structured Latent Generation) → Gaussian Splat + Mesh

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Load a single image + mask (aligned with demo_single_object.ipynb)
IMAGE_FOLDER = SAM3D_NOTEBOOK / "images" / "shutterstock_stylish_kidsroom_1640806567"
MASK_INDEX = 14

image, mask = helper.load_test_image(IMAGE_FOLDER, index=MASK_INDEX)

print(f"Image shape: {image.shape}, dtype: {image.dtype}")
print(f"Mask shape:  {mask.shape}, dtype: {mask.dtype}")
print(f"Mask coverage: {mask.sum() / mask.size * 100:.1f}%")

# Display image + mask overlay
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(image[..., :3])
axes[0].set_title("Input Image")
axes[0].axis("off")
axes[1].imshow(mask, cmap="gray")
axes[1].set_title(f"Object Mask (index={MASK_INDEX})")
axes[1].axis("off")
overlay = image[..., :3].copy()
mask_rgba = np.zeros((*mask.shape, 4), dtype=np.uint8)
mask_rgba[mask] = [255, 0, 0, 128]
axes[2].imshow(overlay)
axes[2].imshow(mask_rgba)
axes[2].set_title("Image + Mask Overlay")
axes[2].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Run full two-stage OpenVINO pipeline (single object)
# Using fewer inference steps for faster demo execution
STAGE1_STEPS = 2
STAGE2_STEPS = 2

# Merge mask into RGBA (same as inference.Inference.merge_mask_to_rgba)
mask_uint8 = mask.astype(np.uint8) * 255
rgba_image = np.concatenate([image[..., :3], mask_uint8[..., None]], axis=-1)
print(f"RGBA image shape: {rgba_image.shape}")

print(f"\nRunning full OpenVINO pipeline (Stage 1 + Stage 2) …")
print(f"  Stage 1 inference steps: {STAGE1_STEPS}")
print(f"  Stage 2 inference steps: {STAGE2_STEPS}")
print(f"  Decode formats: gaussian + mesh")
t0 = time.time()

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    single_output = ov_pipeline.run(
        rgba_image,
        None,
        seed=42,
        stage1_only=False,
        with_mesh_postprocess=False,
        with_texture_baking=False,
        with_layout_postprocess=False,
        use_vertex_color=True,
        stage1_inference_steps=STAGE1_STEPS,
        stage2_inference_steps=STAGE2_STEPS,
    )

elapsed = time.time() - t0
print(f"\nFull pipeline completed in {elapsed:.1f}s")
print(f"Output keys: {list(single_output.keys())}")

# Inspect Gaussian Splat output
if "gs" in single_output and single_output["gs"] is not None:
    gs = single_output["gs"]
    n_points = gs._xyz.shape[0] if hasattr(gs, "_xyz") else "unknown"
    print(f"Gaussian Splat: {n_points:,} points")

# Inspect Mesh output
if "glb" in single_output and single_output["glb"] is not None:
    glb = single_output["glb"]
    print(f"Mesh: {len(glb.vertices):,} vertices, {len(glb.faces):,} faces")
    # Save GLB file
    glb_path = NOTEBOOK_DIR / "single_object.glb"
    glb.export(str(glb_path))
    print(f"Saved GLB → {glb_path}")
else:
    print("No mesh output (mesh decoder may not be available)")

In [ ]:
# Visualize the single-object outputs: Gaussian Splat + Mesh
fig = plt.figure(figsize=(20, 6))

# --- Gaussian Splat ---
if "gs" in single_output and single_output["gs"] is not None:
    gs = single_output["gs"]
    xyz = gs._xyz.detach().cpu().numpy()
    if hasattr(gs, "_features_dc") and gs._features_dc is not None:
        colors = gs._features_dc.detach().cpu().numpy().squeeze()
        colors = (colors - colors.min()) / (colors.max() - colors.min() + 1e-8)
        point_colors = colors[:, :3] if colors.ndim == 2 and colors.shape[1] >= 3 else xyz[:, 1]
    else:
        point_colors = xyz[:, 1]

    ax1 = fig.add_subplot(131, projection="3d")
    ax1.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2], c=point_colors, s=0.5, alpha=0.4)
    ax1.set_xlabel("X"); ax1.set_ylabel("Y"); ax1.set_zlabel("Z")
    ax1.set_title(f"Gaussian Splat ({len(xyz):,} pts)")

# --- Mesh ---
if "glb" in single_output and single_output["glb"] is not None:
    glb = single_output["glb"]
    verts = glb.vertices
    faces = glb.faces

    # Get vertex colors
    if hasattr(glb.visual, "vertex_colors") and glb.visual.vertex_colors is not None:
        vc = glb.visual.vertex_colors[:, :3].astype(np.float32) / 255.0
    else:
        vc = "steelblue"

    ax2 = fig.add_subplot(132, projection="3d")
    ax2.scatter(verts[:, 0], verts[:, 1], verts[:, 2], c=vc, s=0.3, alpha=0.4)
    ax2.set_xlabel("X"); ax2.set_ylabel("Y"); ax2.set_zlabel("Z")
    ax2.set_title(f"Mesh ({len(verts):,} verts, {len(faces):,} faces)")

    ax3 = fig.add_subplot(133)
    ax3.scatter(verts[:, 0], verts[:, 2], s=0.2, alpha=0.3, c=vc if isinstance(vc, np.ndarray) else "steelblue")
    ax3.set_xlabel("X"); ax3.set_ylabel("Z")
    ax3.set_title("Mesh Top-down (XZ)")
    ax3.set_aspect("equal")

plt.tight_layout()
plt.show()

## Multi Object 3D Reconstruction
[back to top ⬆️](#Table-of-contents:)

Following the original [`demo_multi_object.ipynb`](https://github.com/facebookresearch/sam-3d-objects/blob/main/notebook/demo_multi_object.ipynb), we load **multiple** object masks from the same scene and run the OpenVINO pipeline for each object independently.

This demonstrates the ability to reconstruct an entire scene by lifting multiple objects to 3D, matching the multi-object workflow of the original pipeline.

In [ ]:
# Load all available masks from the scene folder
# (aligned with demo_multi_object.ipynb: masks = load_masks(folder, extension=".png"))
all_masks = helper.load_test_masks(IMAGE_FOLDER)
print(f"Found {len(all_masks)} object masks")

# For demo speed, limit to a small subset of masks
MAX_OBJECTS = 2
selected_masks = all_masks[:MAX_OBJECTS]
print(f"Using {len(selected_masks)} masks for multi-object demo")

# Display selected masks
fig, axes = plt.subplots(1, len(selected_masks) + 1, figsize=(6 * (len(selected_masks) + 1), 5))
axes[0].imshow(image[..., :3])
axes[0].set_title("Input Image")
axes[0].axis("off")
for i, m in enumerate(selected_masks):
    axes[i + 1].imshow(m, cmap="gray")
    axes[i + 1].set_title(f"Mask {i}")
    axes[i + 1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Run full OpenVINO pipeline for each object mask
multi_outputs = []
print(f"Running OpenVINO pipeline for {len(selected_masks)} objects …\n")

for i, obj_mask in enumerate(selected_masks):
    mask_u8 = obj_mask.astype(np.uint8) * 255
    rgba = np.concatenate([image[..., :3], mask_u8[..., None]], axis=-1)

    print(f"  Object {i}: mask coverage {obj_mask.sum() / obj_mask.size * 100:.1f}%")
    t0 = time.time()

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        out = ov_pipeline.run(
            rgba,
            None,
            seed=42,
            stage1_only=False,
            with_mesh_postprocess=False,
            with_texture_baking=False,
            with_layout_postprocess=False,
            use_vertex_color=True,
            stage1_inference_steps=STAGE1_STEPS,
            stage2_inference_steps=STAGE2_STEPS,
        )

    elapsed = time.time() - t0
    multi_outputs.append(out)

    n_pts = out["gs"]._xyz.shape[0] if "gs" in out and out["gs"] is not None else 0
    n_verts = len(out["glb"].vertices) if "glb" in out and out["glb"] is not None else 0
    print(f"    → {n_pts:,} Gaussian pts, {n_verts:,} mesh verts in {elapsed:.1f}s")

    # Save each object's GLB
    if "glb" in out and out["glb"] is not None:
        glb_path = NOTEBOOK_DIR / f"multi_object_{i}.glb"
        out["glb"].export(str(glb_path))
        print(f"    → Saved GLB → {glb_path}")

print(f"\nAll {len(multi_outputs)} objects processed.")

In [ ]:
# Visualize multi-object results — each object displayed separately
for i, out in enumerate(multi_outputs):
    fig = plt.figure(figsize=(20, 6))
    fig.suptitle(f"Object {i}", fontsize=14, fontweight="bold")

    # --- Gaussian Splat ---
    if "gs" in out and out["gs"] is not None:
        xyz = out["gs"]._xyz.detach().cpu().numpy()
        if hasattr(out["gs"], "_features_dc") and out["gs"]._features_dc is not None:
            colors = out["gs"]._features_dc.detach().cpu().numpy().squeeze()
            colors = (colors - colors.min()) / (colors.max() - colors.min() + 1e-8)
            pc = colors[:, :3] if colors.ndim == 2 and colors.shape[1] >= 3 else xyz[:, 1]
        else:
            pc = xyz[:, 1]

        ax1 = fig.add_subplot(131, projection="3d")
        ax1.scatter(xyz[:, 0], xyz[:, 1], xyz[:, 2], c=pc, s=0.5, alpha=0.4)
        ax1.set_xlabel("X"); ax1.set_ylabel("Y"); ax1.set_zlabel("Z")
        ax1.set_title(f"Gaussian Splat ({len(xyz):,} pts)")

    # --- Mesh (vertex-colored) ---
    if "glb" in out and out["glb"] is not None:
        glb = out["glb"]
        verts = glb.vertices
        faces = glb.faces
        if hasattr(glb.visual, "vertex_colors") and glb.visual.vertex_colors is not None:
            vc = glb.visual.vertex_colors[:, :3].astype(np.float32) / 255.0
        else:
            vc = "steelblue"

        ax2 = fig.add_subplot(132, projection="3d")
        ax2.scatter(verts[:, 0], verts[:, 1], verts[:, 2], c=vc, s=0.3, alpha=0.4)
        ax2.set_xlabel("X"); ax2.set_ylabel("Y"); ax2.set_zlabel("Z")
        ax2.set_title(f"Mesh ({len(verts):,} verts, {len(faces):,} faces)")

        ax3 = fig.add_subplot(133)
        ax3.scatter(verts[:, 0], verts[:, 2], s=0.2, alpha=0.3,
                    c=vc if isinstance(vc, np.ndarray) else "steelblue")
        ax3.set_xlabel("X"); ax3.set_ylabel("Z")
        ax3.set_title("Mesh Top-down (XZ)")
        ax3.set_aspect("equal")

    plt.tight_layout()
    plt.show()

## Optimize with NNCF Quantization
[back to top ⬆️](#Table-of-contents:)

[NNCF](https://github.com/openvinotoolkit/nncf) provides a suite of advanced algorithms for Neural Network inference optimization in OpenVINO with minimal accuracy drop.

The DINOv2 image encoders are the most computationally expensive components in the SAM-3D-Objects pipeline. We use 8-bit post-training quantization to optimize the **SS DINOv2 image encoder**, which encodes the input image for sparse structure generation.

The optimization process:
1. Create a calibration dataset from COCO128 images
2. Run `nncf.quantize` with Transformer-specific settings
3. Save the quantized INT8 model
4. Validate by comparing FP32 and INT8 outputs

In [ ]:
from notebook_utils import quantization_widget

to_quantize = quantization_widget(False)
to_quantize

In [ ]:
# Fetch skip_kernel_extension for %%skip magic
skip_kernel_extension_file_name = "skip_kernel_extension.py"

if not Path(skip_kernel_extension_file_name).exists():
    r = requests.get(
        url=f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{skip_kernel_extension_file_name}",
    )
    open(skip_kernel_extension_file_name, "w").write(r.text)

%load_ext skip_kernel_extension

### Prepare Calibration Dataset
[back to top ⬆️](#Table-of-contents:)

Download COCO128 images for calibration. Since we only need representative input data to calibrate the quantization parameters, we don't need annotations.

In [ ]:
%%skip not $to_quantize.value

from zipfile import ZipFile
from notebook_utils import download_file

DATA_URL = "https://ultralytics.com/assets/coco128.zip"
OUT_DIR = Path(".")

if not (OUT_DIR / "coco128/images/train2017").exists():
    download_file(DATA_URL, directory=OUT_DIR, show_progress=True)
    with ZipFile("coco128.zip", "r") as zip_ref:
        zip_ref.extractall(OUT_DIR)

In [ ]:
%%skip not $to_quantize.value

import cv2
import nncf
import torch.utils.data as data


class COCOLoader(data.Dataset):
    """Simple COCO image loader for quantization calibration."""

    def __init__(self, images_path):
        self.images = sorted(Path(images_path).glob("*.jpg"))

    def __getitem__(self, index):
        image = cv2.imread(str(self.images[index]))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        # Resize to DINOv2 expected input size and normalize to [0, 1]
        image = cv2.resize(image, (518, 518))
        image = image.astype(np.float32) / 255.0
        # Convert HWC → CHW
        image = np.transpose(image, (2, 0, 1))
        return image

    def __len__(self):
        return len(self.images)


coco_dataset = COCOLoader(OUT_DIR / "coco128/images/train2017")
calibration_loader = data.DataLoader(coco_dataset, batch_size=1, shuffle=False)


def transform_fn(image_data):
    """Extract preprocessed image tensor for quantization calibration."""
    return image_data.numpy()


calibration_dataset = nncf.Dataset(calibration_loader, transform_fn)
print(f"Calibration dataset: {len(coco_dataset)} images")

### Run Quantization
[back to top ⬆️](#Table-of-contents:)

Quantize the SS DINOv2 image encoder using `nncf.quantize` with Transformer-specific settings. This uses a mixed quantization preset (symmetric weights, asymmetric activations) for better accuracy preservation.

> **Note**: Quantization is a time-consuming process. Be patient, it can take several minutes depending on your hardware.

In [ ]:
%%skip not $to_quantize.value

ss_dino_image_int8_path = OV_MODEL_DIR / "ss_dino_image_int8.xml"

if not ss_dino_image_int8_path.exists():
    print("Quantizing SS DINOv2 image encoder …")
    t0 = time.time()

    fp32_model = core.read_model(str(OV_MODEL_DIR / "ss_dino_image.xml"))
    quantized_model = nncf.quantize(
        fp32_model,
        calibration_dataset,
        model_type=nncf.parameters.ModelType.TRANSFORMER,
        subset_size=128,
    )

    ov.save_model(quantized_model, str(ss_dino_image_int8_path))
    elapsed = time.time() - t0
    print(f"Quantization completed in {elapsed:.1f}s")
    print(f"Saved INT8 model → {ss_dino_image_int8_path}")

    # Compare file sizes
    fp32_size = (OV_MODEL_DIR / "ss_dino_image.bin").stat().st_size / (1024 * 1024)
    int8_size = ss_dino_image_int8_path.with_suffix(".bin").stat().st_size / (1024 * 1024)
    print(f"FP32 size: {fp32_size:.1f} MB → INT8 size: {int8_size:.1f} MB ({int8_size / fp32_size * 100:.0f}%)")
else:
    print(f"INT8 model already exists: {ss_dino_image_int8_path}")

### Validate Quantized Model
[back to top ⬆️](#Table-of-contents:)

Compare FP32 and INT8 model outputs to verify the quantized model maintains acceptable accuracy.

In [ ]:
%%skip not $to_quantize.value

# Compare FP32 vs INT8 outputs
test_img = torch.randn(1, 3, 518, 518, dtype=torch.float32)

# FP32 inference
fp32_compiled = core.compile_model(core.read_model(str(OV_MODEL_DIR / "ss_dino_image.xml")), device.value)
fp32_out = torch.from_numpy(fp32_compiled(test_img.numpy())[0].copy())

# INT8 inference
int8_compiled = core.compile_model(core.read_model(str(ss_dino_image_int8_path)), device.value)
int8_out = torch.from_numpy(int8_compiled(test_img.numpy())[0].copy())

# Compare
print("=" * 60)
print("FP32 vs INT8 Comparison (SS DINOv2 Image Encoder)")
print("=" * 60)
max_diff = (fp32_out - int8_out).abs().max().item()
mean_diff = (fp32_out - int8_out).abs().mean().item()
cos_sim = torch.nn.functional.cosine_similarity(
    fp32_out.flatten().unsqueeze(0),
    int8_out.flatten().unsqueeze(0),
).item()

print(f"  Max absolute diff:  {max_diff:.6f}")
print(f"  Mean absolute diff: {mean_diff:.6f}")
print(f"  Cosine similarity:  {cos_sim:.6f}")
print(f"  Status: {'PASS ✓' if cos_sim > 0.99 else 'WARNING — low similarity'}")

## Interactive Gradio Demo
[back to top ⬆️](#Table-of-contents:)

Launch an interactive Gradio demo that allows you to:
- Select an object mask from the demo scene
- Adjust the number of inference steps (fewer steps = faster but lower quality)
- Run the full two-stage 3D reconstruction pipeline
- View the resulting 3D Gaussian Splat visualization

> **Note**: Each reconstruction takes approximately 60–120 seconds on CPU, depending on hardware.

In [ ]:
from gradio_helper import make_demo

demo = make_demo(
    ov_pipeline=ov_pipeline,
    image_folder=IMAGE_FOLDER,
    helper=helper,
    stage1_steps=2,
    stage2_steps=2,
)

# Launch the Gradio demo (non-blocking in notebook mode)
try:
    demo.launch(debug=False, height=800)
except Exception as e:
    print(f"Gradio launch note: {e}")
    print("Run this cell interactively to use the Gradio demo.")